# n-Leaks, predicting confidence heatmap, by parametric gaussian blobs
Predicts parameters for nBlobs gaussian distributions {strength, covar(s1, s1, alpha), mean(x, y)} and the prediction heatmap is then pieced together from these parameters. A 100x100 grid is used to evaluate. The loss is the RMSE for the pixel-wise difference between the predicted field and the true field, which in turn is constructed by taking the leak positions and placing gaussian blobs at each leak location. The blobs are shaped like the wind covariance matrix.

Isn't even able to overfit a small dataset!

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler, random_split
import time
import random
import math
from collections import deque
import numpy as np
import h5py
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gc
import matplotlib.pyplot as plt

from IPython.display import display, clear_output
import ipywidgets as widgets

# load the data
n_samples_to_load = 10
with h5py.File("scene_collections/2025-01-02--16-41-34--X50000.h5", "r") as f:
    data = [{key: np.array(f[item][key]) for key in f[item]} for item in list(f.keys())[:n_samples_to_load]]

print(f"Cuda Check: {torch.cuda.is_available()}")

In [9]:
### plotting classes ---------------------------------------------------------------------------------------------------
class TrainingPlotter:
    """
    Convenience class for adding an interactive training visualization to jupyter notebooks. Does not work with .py!
    """
    
    def __init__(self, n_epochs, l_dataset, mute_output_flag=False):
        
        # initialize the processing relates stats
        self.samp_seen = 0
        self.samp_tota = n_epochs * l_dataset
        self.actual_bs = None
        self.spd_deque = deque(maxlen=10)
        self.spd_avg   = 0
        
        # initialize the control flag that can mute all plotting outputs if set to True
        self.mute_output_flag = mute_output_flag
        
        # some design parameters
        self.padding_factor = 0.03
        self.y_axis_range   = [-0.1, 1.1]
        
        # initialize the plot once: with all the dummy traces and to actually show the figure once
        self.fig_loss = self._initialize_loss_plots()
    
    def _initialize_loss_plots(self):
        if self.mute_output_flag is True: 
            return
        
        # initialize the plotly widget object
        fig_loss = go.FigureWidget()
        
        # add all the dummy traces so that the legend is displayed from the beginning on
        
        # trace [0] training loss variance band upper
        fig_loss.add_scatter(
            x=[None], y=[None], 
            mode="lines", 
            name="upper_training", 
            line=dict(color="firebrick", width=0),
            showlegend=False
            )
        
        # trace [1] training loss variance band lower
        fig_loss.add_scatter(
            x=[None], y=[None], 
            mode="lines", 
            name="train. stdv", 
            line=dict(color="firebrick", width=0), 
            fill="tonexty", 
            fillcolor="rgba(178, 34, 34, 0.14)"
            )
        
        # trace [2] validation loss variance band upper
        fig_loss.add_scatter(
            x=[None], y=[None], 
            mode="lines", 
            name="valid_upper", 
            line=dict(color="seagreen", width=0),
            showlegend=False
            )
        
        # trace [3] validation loss variance band lower
        fig_loss.add_scatter(
            x=[None], y=[None], 
            mode="lines", 
            name="valid. stdv", 
            line=dict(color="seagreen", width=0), 
            fill="tonexty", fillcolor="rgba(46, 139, 87, 0.16)",
            visible="legendonly"
            )
        
        # trace [4] training loss
        fig_loss.add_scatter(
            x=[None], y=[None], 
            mode="lines", 
            name="train. loss", 
            line=dict(color="firebrick", width=2.0)
            )
        
        # trace [5] horizontal line for current best validation loss  
        fig_loss.add_scatter(
            x=[None], y=[None], 
            mode="lines+markers", 
            name="Best val. loss", 
            line=dict(color="seagreen", width=2, dash="dot"), 
            marker=dict(size=[0, 18], opacity=[0, 1], symbol=["circle", "diamond-wide"]), 
            showlegend=False
            )
        
        # trace [6] validation loss (background)
        fig_loss.add_scatter(
            x=[None], y=[None], 
            mode="lines", 
            name="valid. loss", 
            line=dict(color="white", width=3.0),
            showlegend=False,
            legendgroup="group valid"
            )
        
        # trace [7] validation loss
        fig_loss.add_scatter(
            x=[None], y=[None], 
            mode="lines", 
            name="valid. loss", 
            line=dict(color="seagreen", width=2.0),
            legendgroup="group valid"
            )
                
        # layout settings
        fig_loss.update_layout(height=600, plot_bgcolor="white", margin_pad=10,
                        legend=dict(font=dict(family="Lucida Console, monospace", size=14, color="grey")))
        
        fig_loss.update_xaxes(title="number of samples seen (running total over all epochs)",
                        title_standoff=30,
                        title_font_color="grey", title_font_family="Lucida Console, monospace", title_font_size=20, 
                        zeroline=True, zerolinewidth=3, zerolinecolor="gray",
                        showgrid=True, gridwidth=1.5, gridcolor="lightgray", griddash="dash",
                        tickfont=dict(family="Lucida Console, monospace", size=12, color="grey"),
                        range=[-self.padding_factor * self.samp_tota, (1 + self.padding_factor) * self.samp_tota])
        
        fig_loss.update_yaxes(title="loss (distance to leak)",
                        title_standoff=30,  
                        title_font_color="grey", title_font_family="Lucida Console, monospace", title_font_size=20,
                        zeroline=True, zerolinewidth=3, zerolinecolor="gray",
                        showgrid=True, gridwidth = 1.5, gridcolor="lightgray",
                        tickfont=dict(family="Lucida Console, monospace", size=12, color="grey"),
                        range=self.y_axis_range)
        
        # show the plots once. for widget it doesn't seem necessary to use: clear_output(wait=True) and again display(fig)
        display(fig_loss)
        
        return fig_loss


    def update_proc_speed(self, start_time, end_time):
        # calculate and append the current batch speed (samples / second) to the deque list
        batch_time  = end_time - start_time
        batch_speed = self.actual_bs / batch_time
        self.spd_deque.append(batch_speed)
        
        # calculate the new moving average speed (samples / second)
        self.spd_avg = sum(self.spd_deque) / len(self.spd_deque)
            
    def update_val_err_plot(self, loss):
        if self.mute_output_flag is True: 
            return
     
        with self.fig_loss.batch_update():
            
            # updates the validation loss background
            self.fig_loss.data[6].x += (self.samp_seen, )  # Update x-axis for validation trace
            self.fig_loss.data[6].y += (loss.val_avg, )  # Update y-axis for validation trace  
            
            # updates the validation loss trace (all losses over processed samples)
            self.fig_loss.data[7].x += (self.samp_seen, )  # Update x-axis for validation trace
            self.fig_loss.data[7].y += (loss.val_avg, )  # Update y-axis for validation trace
            
            # Update the horizontal line indicator of best validation loss
            self.fig_loss.data[5].x = [0, (1 + self.padding_factor) * self.samp_tota]  # Extend the line across x-axis
            self.fig_loss.data[5].y = [loss.val_best, loss.val_best]  # Same y value for a horizontal line
            
            # validation loss variance upper
            self.fig_loss.data[2].x += (self.samp_seen, )  # Update x-axis for validation trace
            self.fig_loss.data[2].y += (loss.val_avg + loss.val_std_hi, )  # Update y-axis for validation trace 
            
            # validation loss variance lower
            self.fig_loss.data[3].x += (self.samp_seen, )  # Update x-axis for validation trace
            self.fig_loss.data[3].y += (loss.val_avg - loss.val_std_lo, )  # Update y-axis for validation trace     
            
            # Update the annotation showing the current best validation loss
            self.fig_loss.update_layout(
                annotations=[
                    # annotation for the processing speed
                    dict(
                        x=0.5, y=1.1,
                        xref="paper", yref="paper",
                        text=f"Samples/sec (moving avg): <b>{self.spd_avg:.2f}</b>",
                        showarrow=False,
                        font=dict(family="Lucida Console, monospace", size=16, color="gray"),
                        xanchor="center",
                    ),
                    # annotation for the best validation loss
                    dict(
                        x=(1 + self.padding_factor) * self.samp_tota,
                        y=loss.val_best,
                        xref="x", yref="y",
                        text=f"<b> val. best <br> {loss.val_best:09.4f}</b>",
                        showarrow=False,
                        font=dict(family="Lucida Console, monospace", size=14, color="seagreen"),
                        xanchor="left",
                        yanchor="middle",
                        align="left"
                    ),
                ]  
            )      
    
    def update_tra_err_plot(self, loss):
        if self.mute_output_flag is True: 
            return
        
        with self.fig_loss.batch_update():
            # training loss
            self.fig_loss.data[4].x += (self.samp_seen, )  # Update x-axis for training trace
            self.fig_loss.data[4].y += (loss.tra_avg.item(), )  # Update y-axis for training trace 
            
            # training loss variance upper
            self.fig_loss.data[0].x += (self.samp_seen, )  # Update x-axis for training trace
            self.fig_loss.data[0].y += (loss.tra_avg.item() + loss.tra_std_hi.item(), )  # Update y-axis for training trace 
            
            # training loss variance lower
            self.fig_loss.data[1].x += (self.samp_seen, )  # Update x-axis for training trace
            self.fig_loss.data[1].y += (loss.tra_avg.item() - loss.tra_std_lo.item(), )  # Update y-axis for training trace 
                    
            self.fig_loss.update_layout(
                annotations=[
                    # annotation for the processing speed
                    dict(
                        x=0.5, y=1.1,
                        xref="paper", yref="paper",
                        text=f"Samples/sec (moving avg): <b>{self.spd_avg:.2f}</b>",
                        showarrow=False,
                        font=dict(family="Lucida Console, monospace", size=16, color="gray"),
                        xanchor="center",
                    ),
                    # annotation for the current best validation loss
                    dict(
                        x=(1 + self.padding_factor) * self.samp_tota,
                        y=loss.val_best,
                        xref="x", yref="y",
                        text=f"<b> val. best <br> {loss.val_best:09.4f}</b>",
                        showarrow=False,
                        font=dict(family="Lucida Console, monospace", size=14, color="seagreen"),
                        xanchor="left",
                        yanchor="middle",
                        align="left"
                    ),  
                ]
            )
            
class LossStats:
    """
    Convenience saving object for all the loss-statistics that are needed in TrainingPlotter
    """
    
    def __init__(self) -> None:
        # [Scalar, Attached] training loss and its stdv averaged over the current batch
        self.tra_avg    = None
        self.tra_std_hi = None
        self.tra_std_lo = None
                
        # [Scalar, Detached] validation loss and its stdv averaged over the whole validation dataset
        self.val_avg    = None
        self.val_best   = None
        self.val_std_hi = None
        self.val_std_lo = None


# model classes definition ---------------------------------------------------------------------------------------------
class MyDataset(Dataset):
    """
    barebone dataset class
    """
    
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx] 
 
class ScheduledBatchSampler(Sampler):
    """
    batch_sampler with batch size scheduling based on a custom dict. Use as custom batch_sampler fn in Dataloader().
    batch schedule = {how many epochs : batch size, ... : ...}
    """
    
    def __init__(self, data_source: Dataset, schedule: dict, shuffle: bool=False, drop_last: bool=False):
        """
        Args:
            data_source (Dataset): Dataset to sample from.
            schedule (dict)      : mapping batch size to nr of epochs e.g. {128: 10, 256: 20, 512: 50}. no duplicates!
            shuffle (bool)       : whether to shuffle the dataset indices.
            drop_last (bool)     : whether to drop the last incomplete batch.
        """
        
        # set up instance variables
        self.data_source   = data_source                     # this is essentially the whole dataset
        self.num_samples   = len(data_source)                # length of the whole dataset
        self.indices       = list(range(self.num_samples))   # base indices. (shuffled) batches of these are returned
        self.schedule      = self._expand_schedule(schedule) # schedule dict but now with each batch size repeated * n
        self.current_epoch = 0                               # to keep track of the current epoch in training loop
        self.shuffle       = shuffle
        self.drop_last     = drop_last

    def _expand_schedule(self, schedule):
        """
        Expand the schedule to have a list of repeated batch size values. Returns a dict.
        e.g. {128 : 3, 256 : 2} -> {0 : 128, 1 : 128, 2 : 128, 3 : 256, 4 : 256}
        """
        
        expanded_schedule = {}
        cumulative_epochs = 0
        # dict.items() yields a tuple of (key, value)
        for bs, nr_eps in schedule.items():
            for i in range(cumulative_epochs, cumulative_epochs + nr_eps):
                # rebuilds the schedule dict. key i is the epoch nr i and value is the batch size for that epoch
                expanded_schedule[i] = bs
            cumulative_epochs += nr_eps
            
        return expanded_schedule

    def set_epoch(self, epoch):
        """
        update the current epoch when called in the training loop
        note: could also be replaced with something like step() to just increment the epoch by one ...
        """
        
        self.current_epoch = epoch

    def _get_batch_size(self):
        """
        just convenience for getting the currend batch size from the expanded schedule depending on current epoch. If the epoch is larger than what is specified in the schedule, it just returns the last value.
        """
        
        # use dict.get(key) instead of dict[key] in order to be able to specify a default for when key is missing!
        return self.schedule.get(self.current_epoch, list(self.schedule.values())[-1])

    def __iter__(self):
        """
        this function is essentially responsible for returning a generator object. The yield statement is kind of like a return, but instead it just advances the loop further every time the next() method is called, or in a for loop. it kind of remebers the last time it was called.
        """
        
        # determine batch size
        batch_size = self._get_batch_size()
        
        # Shuffle before returning the generator object for the new epoch
        if self.shuffle:
            random.shuffle(self.indices) 

        # determines the start indices for each batch. range(start, stop, step). 
        # Note this is essentially only the index to find the indices in self.indices for one batch xD
        for start_idx in range(0, len(self.indices), batch_size):
            # grab all the indices (now those indices corresponding to samples) belonging to a back
            batch = self.indices[start_idx:start_idx + batch_size]
            # skip yielding the last semi-full batch if self.drop_last is true
            if len(batch) < batch_size and self.drop_last:
                continue
            # basically returns one element when it's called and then pauses the function execution until next()
            yield batch

    def __len__(self):
        """
        just returns the number of batches in the current epoch. changes by 1 depending on whether the last semi-full one is dropped or not.
        """
        
        batch_size = self._get_batch_size()
        if self.drop_last:
            return math.floor(self.num_samples / batch_size)
        else:
            return math.ceil(self.num_samples / batch_size)

class SelfAttention(nn.Module):
    def __init__(self, input_feats_dim: int, latent_dim: int, output_feats_dim: int):
        """
        input_feats are the main data. latent_dim can be chosen.
        -> note, needs skip connection on the outside!
        -> note, this is modified from standard. Normally input_feats_dim = output_feats_dim!
        """
        super().__init__()
        
        # define dimensions
        self.input_feats_dim  = input_feats_dim
        self.latent_dim       = latent_dim
        self.output_feats_dim = output_feats_dim
        
        # define attention weights (= matrices Wq, Wk, Wv)
        self.query = nn.Linear(self.input_feats_dim, self.latent_dim)
        self.key   = nn.Linear(self.input_feats_dim, self.latent_dim)
        self.value = nn.Linear(self.input_feats_dim, self.latent_dim)
        
        # define last linear reshaping layer (= matrix Wout)
        self.linear_out = nn.Linear(self.latent_dim, self.output_feats_dim)
        
        # softmax (row-wise!)
        self.softmax = nn.Softmax(dim=2) 
        
        
    def forward(self, x, msk=None):
        """
        input x  : [BS (var, broadcastable) | input length (var) | feats (known, Architecture defining)]
        input msk: [BS (var, broadcastable) | input length (var) ]
        """

        # fallback for when no mask is provided -> [BS (var, broadcastable) | 1 | feats (known, architecture def.)]
        # (so the maskvector is a lying row vector times batch size)
        if msk is None:
            msk = torch.zeros(x.shape[0], x.shape[1]).to(device)
        
        # calculate the embedded vectors in latent space (= matrices Q, K, V)
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        # calculate the attention matrix -> normalize by latent dimension -> apply softmax -> and weigh it with V
        # mask has [BS, maskvec]. needs unsqueeze(1) so that it is a [BS, 1, maskvec] "lying" row vector -> broadcasted
        SCO = torch.bmm(Q, K.transpose(1, 2)) / (self.latent_dim**0.5)
        SCO = SCO.masked_fill(msk.unsqueeze(1) == 1, float("-inf"))
        ATT = self.softmax(SCO)
        WGH = torch.bmm(ATT, V)
        
        # apply last linear layer to reshape output
        x = self.linear_out(WGH)
        
        return x 
  
class AttentionPooling(nn.Module):
    def __init__(self, input_feats_dim, hidden_layers=[128]):
        """
        Implements the attention pooling mechanism. A simple mlp learns one weight per input sequence. The output is then a linear combination of all the weights and inputs.
        """
        super().__init__()
        
        self.activation = nn.ReLU()
        self.softmax    = nn.Softmax(dim=1)
        
        # set up the sequential mlp to learn the per-input weights [BS | input length | 1]
        layers        = []
        current_feats = input_feats_dim
        for el in hidden_layers:
            layers.append(nn.Linear(current_feats, el))
            layers.append(self.activation)
            current_feats = el
        layers.append(nn.Linear(current_feats, 1))
        self.mlp = nn.Sequential(*layers)
        
    def forward(self, x, msk=None):
        """
        input x:   [BS (var, broadcastable) | input length (var) | n feats (known, architecture defining)]  
        input msk: [BS (var, broadcastable) | input length (var) ]
        """
        
        # fallback for when no mask is provided -> [BS (var, broadcastable) | 1 | feats (known, architecture def.)]
        # (so the maskvector is a lying row vector times batch size)
        if msk is None:
            msk = torch.zeros(x.shape[0], x.shape[1]).to(device) # device is from outside scope!!!
        
        # calculate the weights by learning them with an mlp    
        d = self.mlp(x)
        d = d.masked_fill(msk.unsqueeze(2) == 1, float("-inf"))
        d = self.softmax(d)
        
        # recombine the outputs from a linear combination of weights and inputs
        x = torch.bmm(d.permute(0, 2, 1), x)
        
        return x
    
class WithDoubleAttention(nn.Module):
    def __init__(self) -> None:
        super().__init__()
                
        self.linear_1x1           = nn.Linear( 5, 20)
        self.linear_1x2           = nn.Linear(20, 20)
        self.scene_attention_1    = SelfAttention(20, 20, 20)
        self.linear_1x3           = nn.Linear(20, 20)
        self.scene_attention_2    = SelfAttention(20, 20, 20)
        self.scene_attention_pool = AttentionPooling(20, [20])
        
        self.linear_2x1           = nn.Linear( 20, 100)
        self.linear_2x2           = nn.Linear(100, 100)
        self.time_attention_1     = SelfAttention(100, 100, 100)
        self.linear_2x3           = nn.Linear(100, 100)
        self.time_attention_2     = SelfAttention(100, 100, 100)
        self.time_attention_pool  = AttentionPooling(100, [100])
                
        self.linear_3x1           = nn.Linear(100, 100)
        self.linear_3x2           = nn.Linear(100,  50)
        self.linear_3x3           = nn.Linear( 50, 5*6)
        
        self.activation = nn.ReLU()
        
    def forward(self, sample):   
        # data preparation ---------------------------------------------------------------------------------------------
        # static data
        n_sensors_max = 10
        pos_sensors   = sample["pos_sensors"]
        msk_sensors   = sample["msk_sensors"].unsqueeze(1).repeat(1, 100, 1) # to match the shape for attention layer
        # time series data
        wind_direc    = sample["wind_direc"]
        wind_speed    = sample["wind_speed"]
        con_sensors   = sample["con_sensors"]
        
        # reformat input vector
        input = None
        for t in sample["time"][0]:
            if input is None:
                input = torch.cat((
                    pos_sensors.unsqueeze(1),
                    con_sensors[:, t, :].unsqueeze(1).unsqueeze(3),
                    wind_direc[:, t].unsqueeze(1).unsqueeze(2).unsqueeze(3).expand((-1, 1, n_sensors_max, 1)),
                    wind_speed[:, t].unsqueeze(1).unsqueeze(2).unsqueeze(3).expand((-1, 1, n_sensors_max, 1)),
                ), dim=3)
            else:
                temp = torch.cat((
                    pos_sensors.unsqueeze(1),
                    con_sensors[:, t, :].unsqueeze(1).unsqueeze(3),
                    wind_direc[:, t].unsqueeze(1).unsqueeze(2).unsqueeze(3).expand((-1, 1, n_sensors_max, 1)),
                    wind_speed[:, t].unsqueeze(1).unsqueeze(2).unsqueeze(3).expand((-1, 1, n_sensors_max, 1)),
                ), dim=3)
                input = torch.cat((input, temp), dim=1)
                
        # in the form of [BS, Time, Sensor_nr, features_per_sensor]
        input1 = input.float()
        # in the form of [BS, Time, (Sensor_nr*features_per_sensor)]
        input2 = input1.view(-1, 100, 50)
        
        # select correctly shaped input
        x = input1
        M = msk_sensors
        
        # aggregate all the broadcasting dimensions in one [BS*time, n_sensors, n_feats]
        x = x.reshape(-1, 10, 5)
        M = M.reshape(-1, 10)

        # forward function ---------------------------------------------------------------------------------------------
        
        # scene-elementwise linear layers 
        x = self.activation(self.linear_1x1(x))
        x = self.activation(self.linear_1x2(x))       
        # apply scene-wide attention Layer
        x = self.scene_attention_1(x) + x
        # scene-elementwise linear layers
        x = self.activation(self.linear_1x3(x))
        # apply scene-wide attention Layer
        x = self.scene_attention_2(x) + x
        # apply scene-wide attention pooling (squeeze really important because output is [BS, 1, n_feats=20])
        x = self.scene_attention_pool(x, M).squeeze()
        
        # redistribute time dimension again (now only need to broadcast along BS dimension)
        x = x.reshape(-1, 100, 20)
        
        # time-elementwise linear layers
        x = self.activation(self.linear_2x1(x))
        x = self.activation(self.linear_2x2(x))
        # apply time-wide attention layer
        x = self.time_attention_1(x) + x
        # time-elementwise linear layers
        x = self.activation(self.linear_2x3(x))
        # apply time-wide attention layer
        x = self.time_attention_2(x) + x
        # apply time-wide attention pooling (squeeze really important because output is [BS, 1, n_feats=100])
        x = self.time_attention_pool(x).squeeze()   
        
        # MLP at the end
        x = self.activation(self.linear_3x1(x))
        x = self.activation(self.linear_3x2(x))
        x = self.linear_3x3(x)
        
        return x.view(-1, 5, 6)


### convenience functions ----------------------------------------------------------------------------------------------
def generate_covar(s1: torch.Tensor, s2: torch.Tensor, alpha: torch.Tensor):
    """
    creates valid covariance matrices from 2 eigenvalues (positive!) and one orientation (0-2pi)

    Args:
        s1    (torch.Tensor): first singular value  [BS, nBlobs]
        s2    (torch.Tensor): second singular value [BS, nBlobs]
        alpha (torch.Tensor): angle in rad          [BS, nBlobs]
        
    Returns:
        covar (torch.Tensor): [BS, nBlobs, 2, 2]
    """
    
    # create the diagonal matrix from the singular values => [BS, nBlobs, 2, 2]    
    sigma = torch.concat([s1[:, :, None], s2[:, :, None]], dim=2)
    sigma = torch.diag_embed(sigma)
    
    # create the rotation matrices from angle alpha => [BS, nBlobs, 2, 2]
    sin_alpha = torch.sin(alpha)
    cos_alpha = torch.cos(alpha)
    rotmat    = torch.concat([
        torch.concat([cos_alpha[:, :, None], -sin_alpha[:, :, None]], dim=2)[:, :, :, None],
        torch.concat([sin_alpha[:, :, None],  cos_alpha[:, :, None]], dim=2)[:, :, :, None],
    ], dim=3)
    
    # produce covariance mat by: covar = R.T @ Σ @ R (broadcasting over BS and nBlobs)
    covar = torch.einsum("BNji, BNjk, BNkl -> BNil", rotmat, sigma, rotmat)
    return covar

def batched_2D_gaussian(covar: torch.Tensor, mean: torch.Tensor, stren: torch.Tensor, POS: torch.Tensor, norm = False):
    """
    returns the 2D gaussian (=Z) at position (X, Y) over the whole 2D grid. 
    Uses the formula 1/(2pi * |Σ|**0.5) * exp[-0.5 * (x-μ).T @ Σ^(-1) @ (x-μ)]

    Args:
        covar    (torch.Tensor): [BS, nBlobs, 2, 2]
        mean     (torch.Tensor): [BS, nBlobs, 2]
        strength (torch.Tensor): [BS, nBlobs] 
        POS      (torch.Tensor): [szX, szY, 2]
    """
    
    # subtract mean to get deviations => [BS, nBlobs, szX, szY, 2]
    dev = POS[None, None, :, :, :].to(device) - mean[:, :, None, None, :]
    
    # regularize (for stability) and invert covariances => [BS, nBlobs, 2, 2]
    epsilon   = 10**-5
    covar_inv = covar + epsilon * torch.eye(2).to(device)[None, None, :, :]
    covar_inv = torch.inverse(covar_inv)
    
    # calculate det(covariances) => [BS, nBlobs]
    covar_det = torch.det(covar)
    
    # calculate the exponent (= mahalanobis distance) with dev.T @ Σ.inv @ dev. Watch out, when broadcasting formula, doing dev.T @ Σ.inv @ dev is actually wrong, as for the second @, only the diagonal of the result would be needed! (many unsensible coordinate combinations are calculated in the off-diagonals this way)
    mahalanobi  = torch.einsum("BNxyi, BNij, BNxyj -> BNxy", dev, covar_inv, dev)
    
    if norm is True:
        # calculate the rest of the equation and output normalized gaussians (true pdfs) 
        norm_factor = 1 / (2*torch.pi * torch.sqrt(covar_det))
        pdfs        = torch.einsum("BN, BNxy -> BNxy", norm_factor, torch.exp(-0.5 * mahalanobi))
    else:
        # output non-normalized functions (max of 1)
        pdfs = torch.exp(-0.5 * mahalanobi)
    
    # scale pdfs by strength scalar
    pdfs = torch.einsum("BN, BNxy -> BNxy", stren, pdfs)
        
    return pdfs

def softmaxadd_pdfs(pdfs: torch.Tensor, softmax_temp: float=0.1):
    
    soft_argmax = torch.softmax(pdfs / softmax_temp, dim=1)
    pdfs_added  = torch.einsum("BNxy, BNxy -> Bxy", soft_argmax, pdfs)
    
    return pdfs_added

def my_criterion(sample: dict, pred: torch.Tensor):
    """
    input pred: [BS, n, 6]
    """

    # prepare output grid (POS)  
    grid_lim = [-1, 1]
    grid_res = 100
    x, y     = torch.linspace(grid_lim[0], grid_lim[1], grid_res), torch.linspace(grid_lim[0], grid_lim[1], grid_res)
    X, Y     = torch.meshgrid(x, y, indexing="ij")
    POS      = torch.concat([X[:, :, None], Y[:, :, None]], dim=2)

    # handle PREDICTION --------------------------------------------------------
    # extract all elements
    pred_strength    = pred[:, :, 0]         # [BS, nBlobs]
    pred_mean        = pred[:, :, [1, 2]]    # [BS, nBlobs, 2]
    pred_covar_param = pred[:, :, [3, 4, 5]] # [BS, nBlobs, 3]
    
    # add nonlinearity to raw outputs to ensure positive singular values and positive pred_strength (what about mean??)
    pred_strength                  = torch.nn.functional.sigmoid(pred_strength)
    pred_covar_param[:, :, [0, 1]] = torch.nn.functional.sigmoid(pred_covar_param[:, :, [0, 1]])
    pred_mean                      = torch.clamp(pred_mean, -1, 1)

    # generate all the covariances => [BS, nBlobs, 2, 2]
    pred_covar = generate_covar(pred_covar_param[:, :, 0], pred_covar_param[:, :, 1], 2*np.pi*pred_covar_param[:, :, 2])

    # get PREDICTED pdf over grid and soft-max-add them together (substitute for max)
    pred_pdfs = batched_2D_gaussian(pred_covar, pred_mean, pred_strength, POS)
    pred_pdf  = softmaxadd_pdfs(pred_pdfs, softmax_temp=0.1)
    
    # handle LABELS ------------------------------------------------------------
    # extract all elements
    label_strength = 1 - sample["msk_leaks"]                                                 # [BS, nBlobs]
    label_mean     = sample["pos_leaks"]                                                     # [BS, nBlobs, 2]
    n_max_leaks    = sample["msk_leaks"].shape[1]
    label_covar    = torch.tile(sample["wind_covar"][:, None, :, :], (1, n_max_leaks, 1, 1)) # [BS, nBlobs, 2, 2]
    
    # get LABEL pdf over grid and soft-max-add them together (substitute for max)
    label_pdfs = batched_2D_gaussian(label_covar, label_mean, label_strength, POS)
    label_pdf  = softmaxadd_pdfs(label_pdfs, softmax_temp=0.1)

    # calculate LOSS -----------------------------------------------------------
    diff = label_pdf - pred_pdf
    rmse = torch.sqrt(torch.sum(diff**2, dim=(1, 2))/grid_res**2) 

    return rmse

def do_loss(criterion, sample, pred):
    """
    convenience for calculating loss statistics (returnes ATTACHED training loss stuff)
    """
    
    # calculate euclidean distance from true leak
    loss           = criterion(sample, pred)
    loss_batch_avg = torch.mean(loss)
    
    # calculate the skewed standard deviation
    loss_batch_std_hi = torch.std(loss[loss > loss_batch_avg])
    loss_batch_std_lo = torch.std(loss[loss < loss_batch_avg])
    
    return loss_batch_avg, loss_batch_std_hi, loss_batch_std_lo

def do_validation(criterion, network, validation_dataloader):
    """
    function for calculating validation error (returns DETACHED validation loss stuff)

    Args:
        criterion (_type_): _description_
        network (_type_): _description_
        validation_dataloader (_type_): _description_
    """
    model.eval()
    
    with torch.no_grad():  # Turn off gradient calculations for validation
        
        loss_val = None
        
        for sample in validation_dataloader:
            
            sample = {key: value.to(device) for key, value in sample.items()}
            pred   = network(sample)
    
            if loss_val is None:#
                loss_val = criterion(sample, pred)
            else: 
                loss_val = torch.cat([loss_val, criterion(sample, pred)])
            
        loss_val_avg    = torch.mean(loss_val)
        loss_val_std_hi = torch.std(loss_val[loss_val > loss_val_avg])
        loss_val_std_lo = torch.std(loss_val[loss_val < loss_val_avg])

    return loss_val_avg.item(), loss_val_std_hi.item(), loss_val_std_lo.item()


# set up training and data ---------------------------------------------------------------------------------------------
device             = "cuda:0"

# set batch size scheduling, no duplicate keys! {batch size : how many epochs, ... : ...}
bs_schedule        = {32: 1000}
n_epochs           = sum(bs_schedule.values())

# load dataset and split
dataset            = MyDataset(data)
L                  = len(dataset)
ds_tra, ds_val     = random_split(dataset, [int(0.8 * L), L - int(0.8 * L)])

# setup dataloaders. Train loader is setup with the custom sampler to implement the batch size schedule
sampler_train      = ScheduledBatchSampler(ds_tra, bs_schedule, shuffle=True, drop_last=False)
loader_train       = DataLoader(ds_tra, batch_sampler=sampler_train)
loader_valid       = DataLoader(ds_val, batch_size=len(ds_val), shuffle=False)

# initialize model and move to device
model              = WithDoubleAttention().to(device)
optim              = torch.optim.Adam(model.parameters(), lr=10**-3, betas=(0.9, 0.999), weight_decay=0)

# custom model weight initialization
for m in model.modules():
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

# initialize tracking / counting variables for plotting
loss_stats = LossStats()

# Initialize the plot (input: total epochs, total number of samples in train dataset an optional muting output flag)
PLOTTER = TrainingPlotter(n_epochs, len(ds_tra), mute_output_flag=False)


# Training loop --------------------------------------------------------------------------------------------------------
# do an initial validation and set all the validation-related losses
loss_stats.val_avg, loss_stats.val_std_hi, loss_stats.val_std_lo = do_validation(my_criterion, model, loader_valid)
loss_stats.val_best                                              = loss_stats.val_avg

# update plot
PLOTTER.update_val_err_plot(loss_stats)

for E in range(n_epochs):   

    # activate gradient calculation
    model.train()
    # update the batch size scheduler 
    sampler_train.set_epoch(E)   
        
    for sample in loader_train:
        # get the actual batch size (use some quantity, where batch size can be easily seen)
        PLOTTER.actual_bs  = len(sample["n_sensors"])
        PLOTTER.samp_seen += PLOTTER.actual_bs
        # move batch to device
        sample = {key: value.to(device) for key, value in sample.items()}
        # start timer for batch
        batch_start_time = time.time()
        
        # Zero gradients
        optim.zero_grad()
        # Forward pass
        pred = model(sample)
        # Compute loss(es)
        loss_stats.tra_avg, loss_stats.tra_std_hi, loss_stats.tra_std_lo = do_loss(my_criterion, sample, pred)
        # Compute gradients
        loss_stats.tra_avg.backward()
        # Update weights
        optim.step()
        
        # end timer for batch
        batch_end_time = time.time()
        # update the processing speed and plot
        PLOTTER.update_proc_speed(batch_start_time, batch_end_time)
        PLOTTER.update_tra_err_plot(loss_stats)
        
    # validation at end of each epoch    
    loss_stats.val_avg, loss_stats.val_std_hi, loss_stats.val_std_lo = do_validation(my_criterion, model, loader_valid)
    # check if the new validation loss is the new best one and update if true
    if loss_stats.val_avg < loss_stats.val_best:
        loss_stats.val_best = loss_stats.val_avg
    # update plot
    PLOTTER.update_val_err_plot(loss_stats)
    
    # empty GPU memory, not sure how much this helps
    torch.cuda.empty_cache()




KeyboardInterrupt: 